
### Mildeware 
Middleware in LangChain is a way to control, modify, or add extra behavior to an agent while it is running. It allows you to run custom logic before or after the model/tool execution without changing the main agent code.

Think of it like a middle layer between the user, model, and tools:<br>
Example
 - User → Middleware → Model → Middleware → Tool → Middleware → Response
<br>
uses
- Logging – record what the agent is doing.
- Validation – check input or output.
- Retries – retry when a model/tool fails.
- Guardrails – prevent unwanted actions.
- Human approval – ask a human before executing sensitive tools.
- Summarization – reduce long conversation history



In [1]:

import os

from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")



### Summarization 
Summarization in LangChain means reducing a long conversation or large amount of information into a shorter version while keeping the important points. It helps the AI avoid using too much memory or context when a conversation becomes very long. For example, if a user has a conversation with an agent containing 100 messages, summarization can convert those messages into a short summary containing only the important information.

In [2]:
 
from langchain.agents import create_agent
from langchain.agents.middleware import  SummarizationMiddleware
from langchain_core.messages import  HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model ='gpt-4o-mini',
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model ='gpt-4o-mini',
            trigger = ("messages",10),
            keep =("messages",4)
        )
    ]
)


confi = {"configurable":{"thread_id":"test_1"}}

In [6]:

question = [
    "what is the vlaue of 2x2",
    "what is the vlaue of 22*3",
    "what is the vlaue of 24-2",
    "what is the vlaue of 35/5",
    "what is the vlaue of 62/2",
    "what is the vlaue of 72/3",
    "what is the vlaue of 4+3",
    "what is the vlaue of 9x2",
]

for q in question:
    respone = agent.invoke({"messages":[HumanMessage(content=q)]},confi)
    print("messages: ",respone)
    print("messanage lenthg :",len(respone["messages"]))


messages:  {'messages': [HumanMessage(content='what is the vlaue of 2x2', additional_kwargs={}, response_metadata={}, id='1917666b-cfe7-454b-88ac-677beecb6005'), AIMessage(content='The value of \\(2 \\times 2\\) is \\(4\\).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 18, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_af8966b59d', 'id': 'chatcmpl-EJy8PWJdOL90Xfw6hBzC1QiZlEQ09', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0669f-0ff2-7d51-945c-beb14f75d938-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'inpu

###  Use of same id of thread

- First call:
- thread_id = "test-1"
- User: My name is Ram
-         ↓
- Agent remembers it

- Second call:
- thread_id = "test-1"
- User: What is my name?
-        ↓
- Agent → Ram 



### Token system

In [9]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import  tool
from langchain_core.messages import  HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def find_hotel(city:str)->str:
    """search the search -- return long respone to use more token """
    return f"""
    Hotel in {city}:
    1 . Grand hotel - 5 star, 350$/night,spa,pool,gamming
    2. city Inn - 4 star , 250$/night,businness
    3. stay here -3 star , 100$/night, free wifi
    """

agent = create_agent(
    model = 'gpt-4o-mini',
    tools = [find_hotel],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger = ("tokens",500),
            keep = ("tokens",250)
        )
    ]
)

congfi = {"configurable":{"thread_id":"test_1"}}

In [10]:

def count_token(messages):
    total_token = sum(len(m.content) for m in messages)
    return total_token//4


cities = {"New work","London","New Delhi","Kathmandu","Pokhara"}

for city in cities:
    respone = agent.invoke({"messages":[HumanMessage(content=f"find hotel in {city}")]},congfi)

    tokens = count_token(respone["messages"])
    print(f"city : {tokens}, {len(respone["messages"])} messages")
    print(respone["messages"])
    


city : 148, 4 messages
[HumanMessage(content='find hotel in New work', additional_kwargs={}, response_metadata={}, id='9edf0953-9db0-400d-bb3a-0e25efe43f08'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 56, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_11d07b7a95', 'id': 'chatcmpl-EJyA6XQD4xaDX2I8Kn8NV9hvwWEz9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a066a0-a7cb-7d11-8c93-6162e69038b2-0', tool_calls=[{'name': 'find_hotel', 'args': {'city': 'New work'}, 'id': 'call_Cy4katttIExHbrGxjGipE

### Fraction system 

In [6]:

from langchain.agents import create_agent
from langchain.agents.middleware import  SummarizationMiddleware
from langchain_core.tools import  tool
from langchain_core.messages import  HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def find_hotel(city:str)->str:
    "search hotel in city  - return long respong to use more token"
    return f""" Hotel in {city}:
    1. Grand Hotel - 5 star, 350$/per night,spa,pools,gamming
    2. In city -4 star, 250$/per night,swimming pool
    3. Stay in - 3 star,100$/per night, free wifi
    """

agent = create_agent(
    model='gpt-4o-mini',
    tools = [find_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = 'gpt-4o-mini',
            trigger = ("fraction",0.005),     # 640 tokens
            keep = ("fraction",0.002)         # 256 tokens
        )
    ]
)

confi = {"configurable":{"thread_id":"test-1"}}


In [20]:

def count_token(messages):
    return sum(len(m.content) for m in messages)//4


cities = {"New work","London","New Delhi","Kathmandu","Pokhara"}

for city in cities:
    respone = agent.invoke({'messages':[HumanMessage(content=f"find hotel in {city}")]},confi)

    total_toke = count_token(respone["messages"])
    fraction = total_toke/128000    #gpt-4o-mini limit

    print(f"{city} : {total_toke} tokens, {fraction:.4%}, {len(respone["messages"])} msgs")
    print(respone["messages"])




New work : 139 tokens, 0.1086%, 4 msgs
[HumanMessage(content='find hotel in New work', additional_kwargs={}, response_metadata={}, id='de6e92c1-e90e-46c1-af0f-58984d31d267'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 58, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EJyKR3u4RzZuMjuQme1lE60MmAsRi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a066aa-714b-7253-8ba2-2202b0b7c2a9-0', tool_calls=[{'name': 'find_hotel', 'args': {'city': 'New work'}, 'id': 'call_OQOkh


### HumanInLoopMiddleware
Human-in-the-Loop (HITL) middleware allows an AI agent to pause and ask a human for approval, confirmation, or input before continuing an action.
- ✅ Approve or reject sensitive actions.
- ✏️ Allow humans to modify AI decisions.
- 🔍 Review important outputs before execution.
- 🛑 Stop risky or unwanted actions.
- 🤝 Combine AI automation + human control.

In [3]:

from langchain.agents import create_agent
from langchain.agents.middleware import  HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import  InMemorySaver
from langgraph.types import Command

def read_email_tool(email_id:str)->str:
    """ this is the function of reading email """
    return f"read the gmail by email_id {email_id}"

def send_email_tool(recipant:str,subject:str,body:str)->str:
    "this is function to send the email"
    return f"Email is send to {recipant} with subject {subject} and body {body}"


    

In [4]:
agent = create_agent(
    model = 'gpt-4o',
    tools = [read_email_tool,send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
confi = {"configurable":{"thread_id":"test_approve"}}

In [5]:

respone = agent.invoke({"messages":[HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ")]}
,
confi
)
respone


{'messages': [HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ", additional_kwargs={}, response_metadata={}, id='2491b88d-ee0f-49ba-8219-2ec39a05981e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 99, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-EKgd6HXFBasAVPFJ2nd5mgubw0n5i', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a070d0-e551-7193-83f1-c26e203c8804-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipant':

In [19]:

## approve 

if '__interrupt__' in respone:
    print("⏸️ approving ......")
    respone = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {
                    "type":"approve"
                    }
                ]
            }
        ),
        confi
    )
    print("✅ Result : ",respone["messages"][-1].content)



⏸️ approving ......
✅ Result :  The email has been sent again to rai@123 with the subject "hello" and body "are you fine."


### Reject

In [4]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import  Command

def read_email_tool(email_id:str)->str:
    """ this is the function of reading email """
    return f"read the gmail by email_id {email_id}"

def send_email_tool(recipant:str,subject:str,body:str)->str:
    "this is function to send the email"
    return f"Email is send to {recipant} with subject {subject} and body {body}"


agent = create_agent(
    model = 'gpt-4o',
    tools = [read_email_tool,send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
confi = {"configurable":{"thread_id":"test_reject"}}   

In [5]:
respone = agent.invoke({"messages":[HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ")]}
,
confi
)
respone


{'messages': [HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ", additional_kwargs={}, response_metadata={}, id='79f6c05a-48dc-4095-9781-ef4f0761bd78'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 99, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-EKLdIHACCd0rpfkr9qR3Xxl2FuqZh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06c01-5847-7571-b285-57e5663d497b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipant':

In [6]:

## reject 

if '__interrupt__' in respone:
    print("⏸️ approving ......")
    respone = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {
                        "type":"reject"
                    }
                ]
            }
        ),
        confi
    )
    print("✅ Result : ",respone["messages"][-1].content)
    

⏸️ approving ......
✅ Result :  It looks like you decided not to send the email. If you change your mind or need any other assistance, feel free to let me know!


In [14]:
respone

{'messages': [HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ", additional_kwargs={}, response_metadata={}, id='79f6c05a-48dc-4095-9781-ef4f0761bd78'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 99, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-EKLdIHACCd0rpfkr9qR3Xxl2FuqZh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06c01-5847-7571-b285-57e5663d497b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipant':

### Edition

In [3]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import  Command

def read_email_tool(email_id:str)->str:
    """ this is the function of reading email """
    return f"read the gmail by email_id {email_id}"

def send_email_tool(recipant:str,subject:str,body:str)->str:
    "this is function to send the email"
    return f"Email is send to {recipant} with subject {subject} and body {body}"


agent = create_agent(
    model = 'gpt-4o',
    tools = [read_email_tool,send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
confi = {"configurable":{"thread_id":"test_edit"}}   

In [4]:
respone = agent.invoke({"messages":[HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ")]}
,
confi
)
respone

{'messages': [HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ", additional_kwargs={}, response_metadata={}, id='c1e94fa8-75f1-43b0-b564-b517c3d859fb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 99, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-EKgqm4cKSfpsZEU0nu7DBwVsTr4Lo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a070dd-df71-7410-a01c-a1acfdf57085-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipant':

In [5]:
print(respone["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'recipant': 'rai@123', 'subject': 'hello', 'body': 'are you fine'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'recipant': 'rai@123', 'subject': 'hello', 'body': 'are you fine'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='6cb6ce1be91b2e153203288204bde077')]


In [5]:

## Edition


if '__interrupt__' in respone:
    print("⏸️ resquesting ......")
    respone = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipant":"johncena@gmail.com",
                                "subject":"Namasta",
                                "body":"ilove you bro"
                            }
                        }
                    }
                ]
            }
        ),
       config=confi
    )

    
    


⏸️ resquesting ......


In [6]:
respone

{'messages': [HumanMessage(content="send to this email rai@123 with subject'hello' and body 'are you fine ' ", additional_kwargs={}, response_metadata={}, id='c1e94fa8-75f1-43b0-b564-b517c3d859fb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 99, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-EKgqm4cKSfpsZEU0nu7DBwVsTr4Lo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a070dd-df71-7410-a01c-a1acfdf57085-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool',

In [14]:
respone["messages"][-2].content

'Email is send to johncena@gmail.com with subject Namasta and body ilove you bro'